In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
from sklearn.model_selection import train_test_split
from scipy.sparse import coo_matrix, csr_matrix
from scipy.spatial.distance import jaccard, cosine 
from pytest import approx
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error

In [3]:
MV_users = pd.read_csv('data/users.csv')
MV_movies = pd.read_csv('data/movies.csv')
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [19]:
from collections import namedtuple
Data = namedtuple('Data', ['users','movies','train','test'])
data = Data(MV_users, MV_movies, train, test)
#MV_users.columns
#MV_movies.columns
#train.columns
#test.columns

Index(['uID', 'mID', 'rating'], dtype='object')

In [23]:
from scipy.sparse import coo_matrix
from sklearn.metrics.pairwise import cosine_similarity

#Load your data 
users = pd.read_csv('data/users.csv')
movies = pd.read_csv('data/movies.csv')
train  = pd.read_csv('data/train.csv')  # columns: uID, mID, rating
test   = pd.read_csv('data/test.csv')   # same columns

# Build index mappings from IDs to matrix indices
all_user_ids  = np.union1d(train.uID.unique(),   test.uID.unique())
all_movie_ids = np.union1d(train.mID.unique(),  test.mID.unique())
u2idx = {u: i for i, u in enumerate(all_user_ids)}
m2idx = {m: i for i, m in enumerate(all_movie_ids)}
n_users, n_movies = len(all_user_ids), len(all_movie_ids)

# Assemble the training user×item matrix R
rows = train.uID.map(u2idx.get)
cols = train.mID.map(m2idx.get)
data = train.rating.values
R = coo_matrix((data, (rows, cols)), shape=(n_users, n_movies)).toarray()

# Helper functions
def rmse(preds, actual):
    return np.sqrt(mean_squared_error(actual, preds))

def eval_predictions(pred_matrix):
    # pull out the test‐set predictions
    idx_u = test.uID.map(u2idx.get).values
    idx_m = test.mID.map(m2idx.get).values
    y_true = test.rating.values
    y_pred = pred_matrix[idx_u, idx_m]
    return rmse(y_pred, y_true)

# create baseline
global_mean = train.rating.mean()
pred_global = np.full((n_users, n_movies), global_mean)
print(f"Global‐mean baseline RMSE: {eval_predictions(pred_global):.4f}")

# Item‐item CF (cosine similarity, k=20) (module 3 method)
item_sim = cosine_similarity(R.T)  # shape: (n_movies, n_movies)

def predict_itemcf(R, item_sim, k=20):
    Rhat = np.zeros_like(R, dtype=float)
    for u in range(n_users):
        user_ratings = R[u]
        rated = np.where(user_ratings > 0)[0]
        for m in range(n_movies):
            if rated.size == 0:
                Rhat[u, m] = global_mean
                continue
            sims = item_sim[m, rated]
            topk = np.argsort(sims)[-k:]
            w = sims[topk]
            Rhat[u, m] = (w @ user_ratings[rated[topk]]) / (w.sum() + 1e-8)
    return Rhat

pred_itemcf = predict_itemcf(R, item_sim, k=20)
print(f"Item‐based CF (k=20) RMSE: {eval_predictions(pred_itemcf):.4f}")

# Non‑negative Matrix Factorization (20 components)
nmf = NMF(
    n_components=20,
    init='nndsvd',
    max_iter=200,
    random_state=42
)
W = nmf.fit_transform(np.nan_to_num(R))  # treat missing as zeros
H = nmf.components_
Rhat_nmf = W.dot(H)
print(f"Sklearn NMF (20 factors) RMSE: {eval_predictions(Rhat_nmf):.4f}")


Global‐mean baseline RMSE: 1.1162
Item‐based CF (k=20) RMSE: 0.9385
Sklearn NMF (20 factors) RMSE: 2.8646


I used Item-item cosine similarity as the similarity-based method from module 3. Here I received a RMSE of 0.9385. The sklearn method of non-negative matrix factorization resulted in a much higher RMSE or 2.8646. Some thoughts as to why this RMSE was so much high are:

No bias modeling: Real ratings cluster around each user’s own mean - some users always rate high, others low - and each item’s popularity. A pure non‑negative factorization on raw counts can’t separate out those additive biases.
Additionally, the optimization minimizes ‖R – WH‖² over every cell—observed or not—instead of focusing only on known ratings. There also could have been insufficient hyper‑tuning which without a mask or heavier regularization, the model overfits zeros and underfits the true signal.

Some thoughts as to how it can be fixed are to start, fit only on observed entries, account for user/item biases, and regularize appropriately. Additionally, we could try and use a masked or weighted NMF and also incorporate bias terms explicitly. 